### Pipelining

In [1]:
import numpy as np
import pandas as pd

# Load Data
df = pd.read_csv("cancer_gene_expression.csv")

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Encode Target
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train-Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# Pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline([
    ("scaler", MinMaxScaler()),

    ("feature_selection", SelectKBest(
        score_func=mutual_info_classif,
        k=250   
    )),

    ("model", OneVsRestClassifier(
        RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            max_features=0.2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    ))
])

# Train
pipeline.fit(X_train, y_train)

# Predictions
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)

# Evaluation
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

print("Test Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Test Balanced Accuracy: 0.9192857142857143

Classification Report:

              precision    recall  f1-score   support

        BRCA       0.87      1.00      0.93        60
        COAD       1.00      0.88      0.93        16
        KIRC       1.00      0.90      0.95        30
        LUAD       0.96      0.82      0.88        28
        PRAD       1.00      1.00      1.00        27

    accuracy                           0.94       161
   macro avg       0.97      0.92      0.94       161
weighted avg       0.94      0.94      0.94       161


Confusion Matrix:

[[60  0  0  0  0]
 [ 1 14  0  1  0]
 [ 3  0 27  0  0]
 [ 5  0  0 23  0]
 [ 0  0  0  0 27]]


In [2]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

# Binarize the test labels
y_test_bin = label_binarize(y_test, classes=np.unique(y_encoded))

# Compute ROC-AUC
roc_auc = roc_auc_score(
    y_test_bin,
    y_prob,
    multi_class="ovr",
    average="weighted"   # or "macro"
)

print("\nMulti-class ROC-AUC (OvR):", roc_auc)


Multi-class ROC-AUC (OvR): 0.99680733531188
